In [1]:
!pip install -q opencv-python-headless albumentations timm mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 95.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
import os

BASE_INPUT = '/kaggle/input/datasets'

datasets = {
    'drozy':   f'{BASE_INPUT}/ahmedfaroukksiu/drozy',
    'nthu':    f'{BASE_INPUT}/ikhlaselhamly/nthu-ddd',
    'yawdd':   f'{BASE_INPUT}/enider/yawdd-dataset',
    'cew':     f'{BASE_INPUT}/faisal7/cew-dataset',
    'mrl':     f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset',
    'nitymed': f'{BASE_INPUT}/nikospetrellis/nitymed',
}

BASE = '/kaggle/working/data'
for ds in ['drozy', 'yawdd', 'nitymed']:
    os.makedirs(f'{BASE}/{ds}/frames', exist_ok=True)

print('Dataset Mount Check')
for name, path in datasets.items():
    if os.path.exists(path):
        size = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(path)
            for f in files
        )
        print(f'✅ {name.upper():10} | {size/1e9:.2f} GB | {path}')
    else:
        print(f'❌ {name.upper():10} | NOT FOUND')

Dataset Mount Check
✅ DROZY      | 3.38 GB | /kaggle/input/datasets/ahmedfaroukksiu/drozy
✅ NTHU       | 0.82 GB | /kaggle/input/datasets/ikhlaselhamly/nthu-ddd
✅ YAWDD      | 5.48 GB | /kaggle/input/datasets/enider/yawdd-dataset
✅ CEW        | 0.00 GB | /kaggle/input/datasets/faisal7/cew-dataset
✅ MRL        | 0.08 GB | /kaggle/input/datasets/rameezakther/mrl-eye-open-or-close-dataset
✅ NITYMED    | 0.70 GB | /kaggle/input/datasets/nikospetrellis/nitymed


In [ ]:
import cv2, os, shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm


def extract_frames(video_path, out_dir, label, fps_sample=5, max_frames=500):
    vid_name = os.path.splitext(os.path.basename(video_path))[0]
    save_dir = os.path.join(out_dir, label, vid_name)
    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    interval = max(1, int(fps / fps_sample))
    frame_id, saved = 0, 0
    while saved < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id % interval == 0:
            cv2.imwrite(f'{save_dir}/f{saved:05d}.jpg', frame)
            saved += 1
        frame_id += 1
    cap.release()

def split_videos(records):
    labels = [r['label'] for r in records]
    train, temp = train_test_split(records, test_size=0.30, stratify=labels, random_state=42)
    val, test   = train_test_split(temp, test_size=0.50,
                                   stratify=[r['label'] for r in temp], random_state=42)
    return train, val, test

def split_videos_single_class(records):
    train, temp = train_test_split(records, test_size=0.30, random_state=42)
    val, test   = train_test_split(temp, test_size=0.50, random_state=42)
    return train, val, test

def collect_videos(folder, label):
    exts = {'.avi', '.mp4', '.mkv', '.mov'}
    return [
        {'path': os.path.join(r, f), 'label': label}
        for r, _, files in os.walk(folder)
        for f in files if Path(f).suffix.lower() in exts
    ]

def run_extraction(split_records_map, base_out):
    for split, records in split_records_map.items():
        for r in tqdm(records, desc=f'{base_out.split("/")[-1]}/{split}'):
            extract_frames(r['path'], f'{base_out}/{split}', r['label'])

# ── Clear old frames ──
for ds in ['yawdd', 'drozy', 'nitymed']:
    p = f'{BASE}/{ds}/frames'
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')

# ── YawDD ──
YAWDD_LABEL_MAP = {'Yawning': 'drowsy', 'Normal': 'alert', 'Talking': 'alert'}
yawdd_records = []
for gender in ['Female_mirror', 'Male_mirror Avi Videos']:
    folder = f'{BASE_INPUT}/enider/yawdd-dataset/Mirror/Mirror/{gender}'
    if not os.path.exists(folder):
        print(f'Not found: {folder}')
        continue
    for fname in os.listdir(folder):
        if not fname.endswith('.avi'):
            continue
        label = next((v for k, v in YAWDD_LABEL_MAP.items() if k in fname), None)
        if label:
            yawdd_records.append({'path': os.path.join(folder, fname), 'label': label})

train_v, val_v, test_v = split_videos(yawdd_records)
print(f'YawDD — train:{len(train_v)} val:{len(val_v)} test:{len(test_v)}')
run_extraction({'train': train_v, 'val': val_v, 'test': test_v},
               f'{BASE}/yawdd/frames')

# ── DROZY ──
drozy_records = []
drozy_vid_dir = f'{BASE_INPUT}/ahmedfaroukksiu/drozy/DROZY/videos_i8'
for vid in os.listdir(drozy_vid_dir):
    if not vid.endswith(('.avi', '.mp4', '.mkv')):
        continue
    try:
        session = int(vid.split('-')[1].split('.')[0])
    except:
        print(f'Skipped: {vid}')
        continue
    label = 'alert' if session == 1 else 'drowsy'
    drozy_records.append({'path': os.path.join(drozy_vid_dir, vid), 'label': label})

train_v, val_v, test_v = split_videos(drozy_records)
print(f'DROZY — train:{len(train_v)} val:{len(val_v)} test:{len(test_v)}')
run_extraction({'train': train_v, 'val': val_v, 'test': test_v},
               f'{BASE}/drozy/frames')

# ── NITYMED ──
nitymed_records = collect_videos(
    f'{BASE_INPUT}/nikospetrellis/nitymed/DSM_Dataset-HDTV720', 'drowsy'
)
train_v, val_v, test_v = split_videos_single_class(nitymed_records)
print(f'NITYMED — train:{len(train_v)} val:{len(val_v)} test:{len(test_v)}')
run_extraction({'train': train_v, 'val': val_v, 'test': test_v},
               f'{BASE}/nitymed/frames')

# ── Summary ──
print('\n===== Frame Extraction Summary =====')
for ds in ['yawdd', 'drozy', 'nitymed']:
    for split in ['train', 'val', 'test']:
        for cls in ['alert', 'drowsy']:
            p = f'{BASE}/{ds}/frames/{split}/{cls}'
            if not os.path.exists(p):
                continue
            n_vids   = len(os.listdir(p))
            n_frames = sum(
                len(os.listdir(os.path.join(p, v)))
                for v in os.listdir(p)
                if os.path.isdir(os.path.join(p, v))
            )
            print(f'  {ds:8} | {split:5} | {cls:6} | {n_vids:3} videos | {n_frames:6} frames')

Cleared /kaggle/working/data/yawdd/frames
Cleared /kaggle/working/data/drozy/frames
Cleared /kaggle/working/data/nitymed/frames
YawDD — train:223 val:48 test:48


frames/test: 100%|██████████| 48/48 [00:31<00:00,  1.52it/s]


DROZY — train:25 val:5 test:6


frames/test: 100%|██████████| 6/6 [00:08<00:00,  1.40s/it]


NITYMED — train:88 val:19 test:19


frames/test: 100%|██████████| 19/19 [00:48<00:00,  2.54s/it]


===== Frame Extraction Summary =====
  yawdd    | train | alert  | 144 videos |  22447 frames
  yawdd    | train | drowsy |  79 videos |   9234 frames
  yawdd    | val   | alert  |  31 videos |   4556 frames
  yawdd    | val   | drowsy |  17 videos |   1829 frames
  yawdd    | test  | alert  |  31 videos |   4614 frames
  yawdd    | test  | drowsy |  17 videos |   2037 frames
  drozy    | train | alert  |   8 videos |   4000 frames
  drozy    | train | drowsy |  17 videos |   8500 frames
  drozy    | val   | alert  |   2 videos |   1000 frames
  drozy    | val   | drowsy |   3 videos |   1500 frames
  drozy    | test  | alert  |   2 videos |   1000 frames
  drozy    | test  | drowsy |   4 videos |   2000 frames
  nitymed  | train | drowsy |  88 videos |  17844 frames
  nitymed  | val   | drowsy |  19 videos |   4128 frames
  nitymed  | test  | drowsy |  19 videos |   4365 frames


In [5]:
import cv2, os, gc
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm

# ── Initialize extractor first ──
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request

model_path = '/kaggle/working/face_landmarker.task'
if not os.path.exists(model_path):
    print('Downloading face landmarker model...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task',
        model_path
    )

class FaceRegionExtractor:
    LEFT_EYE  = [33, 160, 158, 133, 153, 144]
    RIGHT_EYE = [362, 385, 387, 263, 373, 380]
    MOUTH     = [61, 291, 39, 181, 0, 17, 269, 405]

    def __init__(self):
        base_options  = python.BaseOptions(model_asset_path=model_path)
        options       = vision.FaceLandmarkerOptions(
            base_options=base_options,
            num_faces=1,
            min_face_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)

    def _crop_region(self, img, landmarks, indices, pad=0.3):
        h, w = img.shape[:2]
        pts  = [(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in indices]
        x1   = max(0, min(p[0] for p in pts) - int(w * pad))
        x2   = min(w, max(p[0] for p in pts) + int(w * pad))
        y1   = max(0, min(p[1] for p in pts) - int(h * pad))
        y2   = min(h, max(p[1] for p in pts) + int(h * pad))
        crop = img[y1:y2, x1:x2]
        return crop if crop.size > 0 else img

    def extract(self, img_rgb):
        import mediapipe as mp
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        results  = self.detector.detect(mp_image)
        if not results.face_landmarks:
            return img_rgb, img_rgb, img_rgb
        lms  = results.face_landmarks[0]
        h, w = img_rgb.shape[:2]
        xs   = [lm.x * w for lm in lms]
        ys   = [lm.y * h for lm in lms]
        x1   = max(0, int(min(xs)) - 20)
        x2   = min(w, int(max(xs)) + 20)
        y1   = max(0, int(min(ys)) - 20)
        y2   = min(h, int(max(ys)) + 20)
        face = img_rgb[y1:y2, x1:x2] if (y2 > y1 and x2 > x1) else img_rgb
        eyes  = self._crop_region(img_rgb, lms, self.LEFT_EYE + self.RIGHT_EYE, pad=0.15)
        mouth = self._crop_region(img_rgb, lms, self.MOUTH, pad=0.2)
        return face, eyes, mouth

extractor = FaceRegionExtractor()
print('Extractor ready')

# ── rest of pre-extraction code below (unchanged) ──
REGIONS_BASE = '/kaggle/working/regions'

def preextract_regions(src_root, dst_root):
    img_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    all_files = [
        os.path.join(r, f)
        for r, _, files in os.walk(src_root)
        for f in files if Path(f).suffix.lower() in img_exts
    ]
    print(f'Pre-extracting {len(all_files)} frames from {src_root}...')
    for src_path in tqdm(all_files):
        rel        = os.path.relpath(src_path, src_root)
        stem       = os.path.splitext(rel)[0]
        face_path  = os.path.join(dst_root, 'face',  stem + '.jpg')
        eye_path   = os.path.join(dst_root, 'eye',   stem + '.jpg')
        mouth_path = os.path.join(dst_root, 'mouth', stem + '.jpg')
        if all(os.path.exists(p) for p in [face_path, eye_path, mouth_path]):
            continue
        os.makedirs(os.path.dirname(face_path),  exist_ok=True)
        os.makedirs(os.path.dirname(eye_path),   exist_ok=True)
        os.makedirs(os.path.dirname(mouth_path), exist_ok=True)
        try:
            img = np.array(Image.open(src_path).convert('RGB'))
        except:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        face, eye, mouth = extractor.extract(img)
        def save(region, path):
            region = cv2.resize(region, (224, 224))
            cv2.imwrite(path, cv2.cvtColor(region, cv2.COLOR_RGB2BGR))
        save(face,  face_path)
        save(eye,   eye_path)
        save(mouth, mouth_path)
    print(f'Done → {dst_root}')
    gc.collect()

# ── Video frames ──
for ds in ['yawdd', 'drozy', 'nitymed']:
    src = f'{BASE}/{ds}/frames'
    dst = f'{REGIONS_BASE}/{ds}'
    if os.path.exists(src):
        preextract_regions(src, dst)

# ── Image datasets ──
img_sources = {
    'mrl_open_train':  f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/train/open_eyes_sample',
    'mrl_close_train': f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/train/close_eyes_sample',
    'mrl_open_test':   f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/test/open_eyes_test',
    'mrl_close_test':  f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/test/close_eyes_test',
    'nthu_notdrowsy':  f'{BASE_INPUT}/ikhlaselhamly/nthu-ddd/NTHU-DDD/notdrowsy',
    'nthu_drowsy':     f'{BASE_INPUT}/ikhlaselhamly/nthu-ddd/NTHU-DDD/drowsy',
    'cew_open':        f'{BASE_INPUT}/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/openEyes',
    'cew_closed':      f'{BASE_INPUT}/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/closedEyes',
}
for name, src in img_sources.items():
    if os.path.exists(src):
        preextract_regions(src, f'{REGIONS_BASE}/images/{name}')

print('\n✅ All regions pre-extracted.')

2026-05-02 14:24:54.735127: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777731894.962678      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777731895.029218      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777731895.516779      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777731895.516818      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777731895.516821      57 computation_placer.cc:177] computation placer alr

Extractor ready


W0000 00:00:1777731914.486171    2210 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777731914.526512    2213 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777731914.548964    2215 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Pre-extracting 44717 frames from /kaggle/working/data/yawdd/frames...


100%|██████████| 44717/44717 [15:49<00:00, 47.08it/s]


...

Done → /kaggle/working/regions/images/nthu_drowsy
Pre-extracting 2462 frames from /kaggle/input/datasets/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/openEyes...


100%|██████████| 2462/2462 [00:29<00:00, 83.13it/s]


Done → /kaggle/working/regions/images/cew_open
Pre-extracting 2384 frames from /kaggle/input/datasets/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/closedEyes...


100%|██████████| 2384/2384 [00:25<00:00, 94.03it/s] 

Done → /kaggle/working/regions/images/cew_closed

✅ All regions pre-extracted.


In [6]:
import shutil

for ds in ['yawdd', 'drozy', 'nitymed']:
    p = f'{BASE}/{ds}/frames'
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Deleted {p}')

# Verify space freed
import subprocess
result = subprocess.run(['du', '-sh', '/kaggle/working'], capture_output=True, text=True)
print(f'Working dir size: {result.stdout}')

Deleted /kaggle/working/data/yawdd/frames
Deleted /kaggle/working/data/drozy/frames
Deleted /kaggle/working/data/nitymed/frames
Working dir size: 5.1G	/kaggle/working



In [7]:
import os, torch, cv2
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import gc

SEQ_LEN    = 16
IMG_SIZE   = 224
BATCH      = 8
WORKERS    = 2
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
REGIONS_BASE = '/kaggle/working/regions'

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.CoarseDropout(num_holes_range=(1,4), hole_height_range=(8,16), hole_width_range=(8,16), p=0.3),
    A.RandomGamma(p=0.3),
    A.RandomShadow(p=0.3),
    A.ImageCompression(quality_range=(50, 90), p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

img_sources = {
    'mrl_open_train':  f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/train/open_eyes_sample',
    'mrl_close_train': f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/train/close_eyes_sample',
    'mrl_open_test':   f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/test/open_eyes_test',
    'mrl_close_test':  f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/test/close_eyes_test',
    'nthu_notdrowsy':  f'{BASE_INPUT}/ikhlaselhamly/nthu-ddd/NTHU-DDD/notdrowsy',
    'nthu_drowsy':     f'{BASE_INPUT}/ikhlaselhamly/nthu-ddd/NTHU-DDD/drowsy',
    'cew_open':        f'{BASE_INPUT}/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/openEyes',
    'cew_closed':      f'{BASE_INPUT}/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/closedEyes',
}

def load_regions(path, transform):
    """Load pre-extracted face/eye/mouth crops directly from disk"""
    blank = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    def load_crop(region_type):
        # Video datasets
        for ds in ['yawdd', 'drozy', 'nitymed']:
            ds_frames = f'{BASE}/{ds}/frames'
            if path.startswith(ds_frames):
                rel = os.path.relpath(path, ds_frames)
                p   = f'{REGIONS_BASE}/{ds}/{region_type}/{os.path.splitext(rel)[0]}.jpg'
                if os.path.exists(p):
                    try:
                        return np.array(Image.open(p).convert('RGB'))
                    except:
                        pass
        # Image datasets
        for name, src in img_sources.items():
            if path.startswith(src):
                rel = os.path.relpath(path, src)
                p   = f'{REGIONS_BASE}/images/{name}/{region_type}/{os.path.splitext(rel)[0]}.jpg'
                if os.path.exists(p):
                    try:
                        return np.array(Image.open(p).convert('RGB'))
                    except:
                        pass
        # Fallback — load original image
        try:
            return np.array(Image.open(path).convert('RGB'))
        except:
            return blank

    def to_tensor(region):
        if region is None or region.size == 0:
            region = blank
        region = cv2.resize(region, (IMG_SIZE, IMG_SIZE))
        return transform(image=region)['image']

    return to_tensor(load_crop('face')), to_tensor(load_crop('eye')), to_tensor(load_crop('mouth'))


# ── Datasets ──
class VideoSequenceDataset(Dataset):
    def __init__(self, records, transform, stride=8):
        self.transform = transform
        self.sequences = []
        for r in records:
            frames = sorted([
                os.path.join(r['path'], f) for f in os.listdir(r['path'])
                if Path(f).suffix.lower() in IMAGE_EXTS
            ])
            if len(frames) < SEQ_LEN:
                continue
            for start in range(0, len(frames) - SEQ_LEN + 1, stride):
                self.sequences.append((frames[start:start + SEQ_LEN], r['label']))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        frame_paths, label = self.sequences[idx]
        faces, eyes, mouths = [], [], []
        for p in frame_paths:
            f, e, m = load_regions(p, self.transform)
            faces.append(f)
            eyes.append(e)
            mouths.append(m)
        return (
            torch.stack(faces),
            torch.stack(eyes),
            torch.stack(mouths),
            torch.tensor(label, dtype=torch.long),
            True
        )

class ImageDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        f, e, m = load_regions(r['path'], self.transform)
        return f, e, m, torch.tensor(r['label'], dtype=torch.long), False


def collect_video_records(split):
    records = []
    for source in ['yawdd', 'drozy']:
        for cls, label in [('alert', 0), ('drowsy', 1)]:
            base = f'{BASE}/{source}/frames/{split}/{cls}'
            if not os.path.exists(base):
                continue
            for vid in os.listdir(base):
                vid_path = os.path.join(base, vid)
                if os.path.isdir(vid_path):
                    records.append({'path': vid_path, 'label': label})
    base = f'{BASE}/nitymed/frames/{split}/drowsy'
    if os.path.exists(base):
        for vid in os.listdir(base):
            vid_path = os.path.join(base, vid)
            if os.path.isdir(vid_path):
                records.append({'path': vid_path, 'label': 1})
    return records

def collect_image_records():
    source_list = [
        (img_sources['mrl_open_train'],  0, 'mrl'),
        (img_sources['mrl_close_train'], 1, 'mrl'),
        (img_sources['mrl_open_test'],   0, 'mrl'),
        (img_sources['mrl_close_test'],  1, 'mrl'),
        (img_sources['nthu_notdrowsy'],  0, 'nthu'),
        (img_sources['nthu_drowsy'],     1, 'nthu'),
        (img_sources['cew_open'],        0, 'cew'),
        (img_sources['cew_closed'],      1, 'cew'),
    ]
    records = []
    for folder, label, source in source_list:
        if not os.path.exists(folder):
            print(f'⚠️  Missing: {folder}')
            continue
        for f in Path(folder).rglob('*'):
            if f.suffix.lower() in IMAGE_EXTS:
                records.append({'path': str(f), 'label': label, 'source': source})
    return records

def collate_fn(batch):
    faces, eyes, mouths, labels, is_seqs = zip(*batch)
    return list(faces), list(eyes), list(mouths), torch.stack(labels), list(is_seqs)

# ── Build splits ──
train_vid = collect_video_records('train')
val_vid   = collect_video_records('val')
test_vid  = collect_video_records('test')

img_records = collect_image_records()
img_labels  = [r['label'] for r in img_records]
img_train, img_temp = train_test_split(img_records, test_size=0.30,
                                        stratify=img_labels, random_state=42)
img_val, img_test   = train_test_split(img_temp, test_size=0.50,
                                        stratify=[r['label'] for r in img_temp],
                                        random_state=42)

# ── Build Datasets ──
train_ds = ConcatDataset([
    VideoSequenceDataset(train_vid, train_transform, stride=8),
    ImageDataset(img_train, train_transform)
])
val_ds = ConcatDataset([
    VideoSequenceDataset(val_vid, val_transform, stride=16),
    ImageDataset(img_val, val_transform)
])
test_ds = ConcatDataset([
    VideoSequenceDataset(test_vid, val_transform, stride=16),
    ImageDataset(img_test, val_transform)
])

del train_vid, val_vid, test_vid, img_train, img_val, img_test, img_records
gc.collect()

# ── Build Loaders ──
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=WORKERS, pin_memory=False, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=WORKERS, pin_memory=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          num_workers=WORKERS, pin_memory=False, collate_fn=collate_fn)

print(f'Train samples : {len(train_ds)}')
print(f'Val   samples : {len(val_ds)}')
print(f'Test  samples : {len(test_ds)}')
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

Train samples : 30692
Val   samples : 6577
Test  samples : 6577
Train batches : 3837
Val   batches : 823
Test  batches : 823


In [8]:
import torch
import torch.nn as nn
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

class TemporalAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden, hidden // 4),
            nn.Tanh(),
            nn.Linear(hidden // 4, 1)
        )

    def forward(self, x):
        # x: (B, T, hidden)
        weights = torch.softmax(self.attn(x), dim=1)  # (B, T, 1)
        return (x * weights).sum(dim=1)                # (B, hidden)


class MultiRegionCNNTCN(nn.Module):
    def __init__(self, backbone='tf_efficientnet_lite2', hidden=256, num_classes=2, seq_len=16):
        super().__init__()
        self.seq_len = seq_len

        # Separate backbone per region
        self.cnn_face  = timm.create_model(backbone, pretrained=True, num_classes=0)
        self.cnn_eye   = timm.create_model(backbone, pretrained=True, num_classes=0)
        self.cnn_mouth = timm.create_model(backbone, pretrained=True, num_classes=0)
        feat_dim = self.cnn_face.num_features  # 1408 for lite2

        # Region attention gate — learns which region matters most
        self.region_attn = nn.Sequential(
            nn.Linear(feat_dim * 3, 3),
            nn.Softmax(dim=1)
        )

        # Fuse 3 regions → hidden
        self.fusion = nn.Sequential(
            nn.Linear(feat_dim * 3, hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # TCN for temporal modelling
        self.tcn = nn.Sequential(
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=4, dilation=4),
            nn.BatchNorm1d(hidden), nn.ReLU(),
        )

        # Temporal attention — focus on most drowsy frames
        self.temporal_attn = TemporalAttention(hidden)

        # Single image head
        self.image_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Linear(hidden, num_classes)

    def extract_regions(self, face, eye, mouth):
        """Extract and fuse features from all 3 regions — (B, hidden)"""
        f_feat = self.cnn_face(face)
        e_feat = self.cnn_eye(eye)
        m_feat = self.cnn_mouth(mouth)

        combined = torch.cat([f_feat, e_feat, m_feat], dim=1)  # (B, feat_dim*3)

        # Weighted attention per region
        weights  = self.region_attn(combined)                   # (B, 3)
        weighted = torch.cat([
            f_feat * weights[:, 0:1],
            e_feat * weights[:, 1:2],
            m_feat * weights[:, 2:3],
        ], dim=1)                                               # (B, feat_dim*3)

        return self.fusion(weighted)                            # (B, hidden)

    def forward(self, face, eye, mouth, is_sequence=True):
        if is_sequence:
            B, T, C, H, W = face.shape

            feats = self.extract_regions(
                face.view(B*T, C, H, W),
                eye.view(B*T, C, H, W),
                mouth.view(B*T, C, H, W)
            )                                                   # (B*T, hidden)

            feats = feats.view(B, T, -1).permute(0, 2, 1)      # (B, hidden, T)
            feats = self.tcn(feats)                             # (B, hidden, T)
            feats = feats.permute(0, 2, 1)                      # (B, T, hidden)
            feats = self.temporal_attn(feats)                   # (B, hidden)
        else:
            feats = self.extract_regions(face, eye, mouth)
            feats = self.image_head(feats)

        return self.classifier(feats)                           # (B, num_classes)


model = MultiRegionCNNTCN(seq_len=SEQ_LEN).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

# ── Sanity check ──
with torch.no_grad():
    dummy_face = torch.randn(2, SEQ_LEN, 3, 224, 224).to(device)
    dummy_eye  = torch.randn(2, SEQ_LEN, 3, 224, 224).to(device)
    dummy_mouth= torch.randn(2, SEQ_LEN, 3, 224, 224).to(device)
    out = model(dummy_face, dummy_eye, dummy_mouth, is_sequence=True)
    print(f'Video path output : {out.shape}')

    dummy_face2 = torch.randn(2, 3, 224, 224).to(device)
    dummy_eye2  = torch.randn(2, 3, 224, 224).to(device)
    dummy_mouth2= torch.randn(2, 3, 224, 224).to(device)
    out2 = model(dummy_face2, dummy_eye2, dummy_mouth2, is_sequence=False)
    print(f'Image path output : {out2.shape}')

Using: cuda


model.safetensors:   0%|          | 0.00/24.6M [00:00<?, ?B/s]

Params: 16.1M
Video path output : torch.Size([2, 2])
Image path output : torch.Size([2, 2])


In [9]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import gc

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

ACCUM_STEPS = 4
EPOCHS      = 10
EARLY_STOP  = 4

best_val_acc = 0
no_improve   = 0

train_loss_history = []
train_acc_history  = []
val_loss_history   = []
val_acc_history    = []

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    optimizer.zero_grad()

    for step, (faces, eyes, mouths, labels, is_seqs) in enumerate(loader):
        labels  = labels.to(device)
        seq_idx = [i for i, s in enumerate(is_seqs) if s]
        img_idx = [i for i, s in enumerate(is_seqs) if not s]
        all_logits, all_lbls = [], []

        with torch.set_grad_enabled(train):
            if seq_idx:
                f = torch.stack([faces[i]  for i in seq_idx]).to(device)
                e = torch.stack([eyes[i]   for i in seq_idx]).to(device)
                m = torch.stack([mouths[i] for i in seq_idx]).to(device)
                all_logits.append(model(f, e, m, is_sequence=True))
                all_lbls.append(labels[seq_idx])
                del f, e, m

            if img_idx:
                f = torch.stack([faces[i]  for i in img_idx]).to(device)
                e = torch.stack([eyes[i]   for i in img_idx]).to(device)
                m = torch.stack([mouths[i] for i in img_idx]).to(device)
                all_logits.append(model(f, e, m, is_sequence=False))
                all_lbls.append(labels[img_idx])
                del f, e, m

            logits = torch.cat(all_logits)
            lbls   = torch.cat(all_lbls)
            loss   = criterion(logits, lbls) / ACCUM_STEPS

        if train:
            loss.backward()
            if (step + 1) % ACCUM_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS
        correct    += (logits.argmax(1) == lbls).sum().item()
        total      += len(lbls)
        del logits, lbls, loss

        if step % 100 == 0:
            gc.collect()

        if train and step % 200 == 0:
            pct = step / len(loader) * 100
            print(f'  {pct:5.1f}% [{step:4d}/{len(loader)}] '
                  f'loss: {total_loss/(step+1):.4f}  '
                  f'acc: {correct/total:.4f}')

    return total_loss / len(loader), correct / total


# ── Training loop ──
for epoch in range(1, EPOCHS + 1):
    print(f'\n{"═"*60}')
    print(f'  Epoch {epoch:02d}/{EPOCHS}  |  LR: {optimizer.param_groups[0]["lr"]:.2e}')
    print(f'{"═"*60}')

    train_loss, train_acc = run_epoch(train_loader, train=True)
    gc.collect()

    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    print(f'  ► Train | Loss: {train_loss:.4f}  Acc: {train_acc:.4f}')

    if epoch % 2 == 0:
        print(f'  ► Val   | running...')
        val_loss, val_acc = run_epoch(val_loader, train=False)
        gc.collect()

        val_loss_history.append(val_loss)
        val_acc_history.append(val_acc)
        print(f'  ► Val   | Loss: {val_loss:.4f}  Acc: {val_acc:.4f}')

        scheduler.step(val_acc)
        print(f'  ► LR now → {optimizer.param_groups[0]["lr"]:.2e}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve   = 0
            torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
            print(f'  ✓ New best saved (val_acc={val_acc:.4f})')
        else:
            no_improve += 1
            print(f'  ✗ No improvement ({no_improve}/{EARLY_STOP})')
            if no_improve >= EARLY_STOP:
                print(f'  ⚠ Early stopping triggered.')
                break

print(f'\n{"═"*60}')
print(f'  Training complete.  Best val_acc: {best_val_acc:.4f}')
print(f'{"═"*60}')


════════════════════════════════════════════════════════════
  Epoch 01/10  |  LR: 1.00e-04
════════════════════════════════════════════════════════════
    0.0% [   0/3837] loss: 0.6975  acc: 0.5000
    5.2% [ 200/3837] loss: 0.6676  acc: 0.6107
   10.4% [ 400/3837] loss: 0.5761  acc: 0.6886
   15.6% [ 600/3837] loss: 0.5255  acc: 0.7219
   20.8% [ 800/3837] loss: 0.4890  acc: 0.7453
   26.1% [1000/3837] loss: 0.4616  acc: 0.7641
   31.3% [1200/3837] loss: 0.4389  acc: 0.7780
   36.5% [1400/3837] loss: 0.4223  acc: 0.7880
   41.7% [1600/3837] loss: 0.4092  acc: 0.7964
   46.9% [1800/3837] loss: 0.3962  acc: 0.8043
   52.1% [2000/3837] loss: 0.3850  acc: 0.8116
   57.3% [2200/3837] loss: 0.3761  acc: 0.8171
   62.5% [2400/3837] loss: 0.3694  acc: 0.8216
   67.8% [2600/3837] loss: 0.3616  acc: 0.8264
   73.0% [2800/3837] loss: 0.3545  acc: 0.8306
   78.2% [3000/3837] loss: 0.3473  acc: 0.8350
   83.4% [3200/3837] loss: 0.3393  acc: 0.8393
   88.6% [3400/3837] loss: 0.3349  acc: 0.8419


In [11]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import numpy as np
import torch

# ── Load best model ──
model.load_state_dict(torch.load('/kaggle/working/best_model.pth', map_location=device))
model.eval()

# ── Run inference on test set ──
all_preds, all_labels = [], []

with torch.no_grad():
    for step, (faces, eyes, mouths, labels, is_seqs) in enumerate(test_loader):
        labels  = labels.to(device)
        seq_idx = [i for i, s in enumerate(is_seqs) if s]
        img_idx = [i for i, s in enumerate(is_seqs) if not s]
        all_logits, all_lbls = [], []

        if seq_idx:
            f = torch.stack([faces[i]  for i in seq_idx]).to(device)
            e = torch.stack([eyes[i]   for i in seq_idx]).to(device)
            m = torch.stack([mouths[i] for i in seq_idx]).to(device)
            all_logits.append(model(f, e, m, is_sequence=True))
            all_lbls.append(labels[seq_idx])
            del f, e, m

        if img_idx:
            f = torch.stack([faces[i]  for i in img_idx]).to(device)
            e = torch.stack([eyes[i]   for i in img_idx]).to(device)
            m = torch.stack([mouths[i] for i in img_idx]).to(device)
            all_logits.append(model(f, e, m, is_sequence=False))
            all_lbls.append(labels[img_idx])
            del f, e, m

        logits = torch.cat(all_logits)
        lbls   = torch.cat(all_lbls)
        preds  = logits.argmax(1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(lbls.cpu().tolist())

        if step % 50 == 0:
            print(f'  Evaluating step {step}/{len(test_loader)}')

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ================================================================
# 1. Classification Report
# ================================================================
print('\n========== Classification Report ==========')
print(classification_report(
    all_labels, all_preds,
    target_names=['Alert', 'Drowsy'],
    digits=4
))

# ================================================================
# 2. Confusion Matrix
# ================================================================
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Alert', 'Drowsy'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()
print('Saved → /kaggle/working/confusion_matrix.png')

# ================================================================
# 3. Training & Validation Loss / Accuracy over epochs
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_trained = list(range(1, len(train_loss_history) + 1))
val_epochs     = list(range(2, len(train_loss_history) + 1, 2))

axes[0].plot(epochs_trained, train_loss_history, 'b-o', label='Train Loss', markersize=4)
axes[0].plot(val_epochs,     val_loss_history,   'r-o', label='Val Loss',   markersize=4)
axes[0].set_title('Loss over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_trained, train_acc_history, 'b-o', label='Train Acc', markersize=4)
axes[1].plot(val_epochs,     val_acc_history,   'r-o', label='Val Acc',   markersize=4)
axes[1].set_title('Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=150)
plt.show()
print('Saved → /kaggle/working/training_history.png')

  Evaluating step 0/823
  Evaluating step 50/823
  Evaluating step 100/823
  Evaluating step 150/823
  Evaluating step 200/823
  Evaluating step 250/823
  Evaluating step 300/823
  Evaluating step 350/823
  Evaluating step 400/823
  Evaluating step 450/823
  Evaluating step 500/823
  Evaluating step 550/823
  Evaluating step 600/823
  Evaluating step 650/823
  Evaluating step 700/823
  Evaluating step 750/823
  Evaluating step 800/823

========== Classification Report ==========
              precision    recall  f1-score   support

       Alert     0.9695    0.9824    0.9759      3295
      Drowsy     0.9821    0.9689    0.9755      3282

    accuracy                         0.9757      6577
   macro avg     0.9758    0.9757    0.9757      6577
weighted avg     0.9758    0.9757    0.9757      6577

Saved → /kaggle/working/confusion_matrix.png
Saved → /kaggle/working/training_history.png
